# Fine-Tuning BERT on SST-2 with Attention Weight Visualization

This notebook provides an end-to-end NLP workflow:
1. **Pre-flight verification:** Ensures all required dependencies are available before execution.
2. **Dataset preprocessing:** Loads the SST-2 dataset from GLUE with dynamic batch padding.
3. **Fine-tuning:** Fine-tunes `bert-base-uncased` for binary sentiment classification using Hugging Face `Trainer`.
4. **Attention Extraction:** Captures multi-head attention matrices across all 12 Transformer layers using eager attention.
5. **Visualization:** Generates static 2D attention heatmaps, a `[CLS]` token importance bar chart, and optional interactive BertViz views.

## 1. Environment Verification
Check that required libraries are installed. To install dependencies, run `pip install -r requirements.txt` in your activated virtual environment.

In [ ]:
import importlib.util
import platform

required_packages = [
    'torch', 'transformers', 'datasets', 'accelerate', 
    'sklearn', 'matplotlib', 'seaborn'
]

missing = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing:
    raise RuntimeError(
        f"Missing required packages: {missing}.\n"
        "Please install them using: python -m pip install -r requirements.txt"
    )

print(f"Python Version: {platform.python_version()}")
for pkg in required_packages:
    mod = importlib.import_module(pkg)
    print(f"  - {pkg}: {getattr(mod, '__version__', 'installed')}")

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Runtime Device: {device}")

## 2. Configuration & Dataset Preprocessing

- **Dynamic Padding:** We tokenize sentences with truncation enabled, leaving padding to `DataCollatorWithPadding` for optimal memory and compute efficiency.
- **`FAST_RUN` Flag:** Set `FAST_RUN = True` explicitly for a smoke test (500 train, 100 eval samples). It is `False` by default on every device.

In [ ]:
import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, set_seed

# --- Experiment Configuration ---
SEED = 42
MAX_LENGTH = 64

# Set FAST_RUN = True for a rapid 1-minute smoke test (500 train / 100 eval samples).
# The default is identical on CPU and GPU for comparable experiments.
FAST_RUN = False
TRAIN_LIMIT = 500
EVAL_LIMIT = 100

# Set deterministic seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print(f"Target device: {device}")
print(f"FAST_RUN mode: {FAST_RUN} (train samples: {TRAIN_LIMIT if FAST_RUN else 'Full'}, eval samples: {EVAL_LIMIT if FAST_RUN else 'Full'})")

# 1. Load SST-2 dataset
DATASET_ID = 'nyu-mll/glue'
DATASET_REVISION = 'bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c'
MODEL_CHECKPOINT = 'google-bert/bert-base-uncased'
MODEL_REVISION = '86b5e0934494bd15c9632b12f734a8a67f723594'
raw_datasets = load_dataset(DATASET_ID, 'sst2', revision=DATASET_REVISION)
train_data = raw_datasets['train'].shuffle(seed=SEED)
eval_data = raw_datasets['validation']

if FAST_RUN:
    train_data = train_data.select(range(min(TRAIN_LIMIT, len(train_data))))
    eval_data = eval_data.select(range(min(EVAL_LIMIT, len(eval_data))))

# 2. Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, revision=MODEL_REVISION)

def tokenize_fn(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=MAX_LENGTH)

# 3. Apply tokenization (dropping raw text columns to streamline tensors)
train_dataset = train_data.map(tokenize_fn, batched=True, remove_columns=['sentence', 'idx'])
eval_dataset = eval_data.map(tokenize_fn, batched=True, remove_columns=['sentence', 'idx'])

# 4. Dynamic batch collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f"Prepared training samples: {len(train_dataset):,}, validation samples: {len(eval_dataset):,}")

## 3. Model Initialization & Fine-Tuning

Important architectural configurations:
- `output_attentions=False`: Kept off during training to prevent memory bloating and backpropagation overhead.
- `id2label` / `label2id`: Explicitly mapped to `{0: 'Negative', 1: 'Positive'}` for clean model metadata.
- `save_total_limit=2`: Restricts saved checkpoints to prevent multi-gigabyte disk usage.

In [ ]:
import inspect
import json
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

id2label = {0: 'Negative', 1: 'Positive'}
label2id = {'Negative': 0, 'Positive': 1}

# 1. Initialize model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, revision=MODEL_REVISION,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    output_attentions=False  # Keep disabled during training
)

# 2. Metrics evaluation function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': float(acc),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1)
    }

# 3. Training arguments setup
epochs = 1 if FAST_RUN else 2
batch_size = 16

training_args_candidates = {
    'output_dir': './runs/bert-sst2-v2/checkpoints',
    'learning_rate': 2e-5,
    'per_device_train_batch_size': batch_size,
    'per_device_eval_batch_size': batch_size * 2,
    'num_train_epochs': epochs,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'eval_strategy': 'epoch',
    'evaluation_strategy': 'epoch',
    'save_strategy': 'epoch',
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1',
    'greater_is_better': True,
    'logging_steps': 10 if FAST_RUN else 50,
    'fp16': torch.cuda.is_available(),
    'seed': SEED,
    'data_seed': SEED,
    'report_to': 'none'
}

# Filter arguments supported by installed transformers version
ta_sig = inspect.signature(TrainingArguments.__init__).parameters
if 'eval_strategy' in ta_sig and 'evaluation_strategy' in training_args_candidates:
    del training_args_candidates['evaluation_strategy']
valid_ta_args = {k: v for k, v in training_args_candidates.items() if k in ta_sig}
training_args = TrainingArguments(**valid_ta_args)

# 4. Trainer configuration
trainer_candidates = {
    'model': model,
    'args': training_args,
    'train_dataset': train_dataset,
    'eval_dataset': eval_dataset,
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
    'processing_class': tokenizer,
    'tokenizer': tokenizer
}
tr_sig = inspect.signature(Trainer.__init__).parameters
if 'processing_class' in tr_sig and 'tokenizer' in trainer_candidates:
    del trainer_candidates['tokenizer']
valid_tr_args = {k: v for k, v in trainer_candidates.items() if k in tr_sig}
trainer = Trainer(**valid_tr_args)

# 5. Execute training & evaluation
print(f"--> Training on {device} ({len(train_dataset)} samples, {epochs} epoch(s))...")
trainer.train()

print("--> Evaluating on validation split...")
eval_results = trainer.evaluate()
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# 6. Save final model and tokenizer
SAVE_DIRECTORY = './runs/bert-sst2-v2/model'
trainer.save_model(SAVE_DIRECTORY)
tokenizer.save_pretrained(SAVE_DIRECTORY)
trainer.save_metrics('eval', eval_results)
trainer.save_state()
run_metadata = {
    'model_id': MODEL_CHECKPOINT, 'model_revision': MODEL_REVISION,
    'dataset_id': DATASET_ID, 'dataset_revision': DATASET_REVISION,
    'seed': SEED, 'fast_run': FAST_RUN, 'epochs': epochs,
    'batch_size': batch_size, 'max_length': MAX_LENGTH
}
os.makedirs(SAVE_DIRECTORY, exist_ok=True)
with open(os.path.join(SAVE_DIRECTORY, 'run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2)
print(f"--> Model and artifacts saved to '{SAVE_DIRECTORY}'.")

## 4. Attention Weight Extraction & Prediction

- We release the trainer from memory before loading the inference model.
- The model is loaded with `attn_implementation='eager'` to guarantee attention matrix outputs.
- Softmax is applied to raw logits to compute normalized confidence probabilities.

In [ ]:
import gc

# Free training model memory
del trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load model specifically for attention extraction
inference_model = AutoModelForSequenceClassification.from_pretrained(
    SAVE_DIRECTORY,
    attn_implementation='eager'
).to(device)
inference_model.eval()

# Test on a contrastive sentence (positive clause followed by a negative adversative clause)
sample_sentence = "The movie was absolutely fantastic, but the acting was terrible."
inputs = tokenizer(sample_sentence, return_tensors='pt', truncation=True, max_length=MAX_LENGTH).to(device)

with torch.inference_mode():
    outputs = inference_model(**inputs, output_attentions=True)

logits = outputs.logits
probs = torch.softmax(logits, dim=-1)[0].cpu()
pred_id = int(probs.argmax())
pred_label = inference_model.config.id2label[pred_id]

print(f"Input Text: '{sample_sentence}'")
print(f"Predicted Sentiment: {pred_label} (Predicted probability: {probs[pred_id]:.2%})")
print(f"  - Negative probability: {probs[0]:.2%}")
print(f"  - Positive probability: {probs[1]:.2%}")

# Tokens and attention tensors
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
cpu_attentions = tuple(layer.cpu() for layer in outputs.attentions)

print(f"\nExtracted {len(cpu_attentions)} transformer layers.")
print(f"Attention tensor shape per layer: {cpu_attentions[0].shape}")
print(f"  [batch_size={cpu_attentions[0].shape[0]}, heads={cpu_attentions[0].shape[1]}, tokens={cpu_attentions[0].shape[2]}]")

## 5. Attention Visualizations (Static Heatmap & [CLS] Importance)

We provide two complementary static visualizations:
1. **Full Attention Matrix:** Heatmap showing how every token attends to every other token for a given layer and head.
2. **`[CLS]` Token Attention Distribution:** Bar chart showing how the classification token distributes its attention across the input tokens in the final layer.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_attention_matrix(attentions, tokens, layer_idx=11, head_idx=0):
    """Plots a 2D heatmap of attention weights between all token pairs."""
    matrix = attentions[layer_idx][0, head_idx].numpy()
    
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        matrix, 
        xticklabels=tokens, 
        yticklabels=tokens, 
        cmap="Blues", 
        annot=(len(tokens) <= 15),
        fmt=".2f",
        vmin=0.0,
        vmax=1.0
    )
    plt.title(f"Attention Weights: Layer {layer_idx + 1}, Head {head_idx + 1}", fontsize=13, pad=12)
    plt.xlabel("Key Tokens (Attended To)", fontsize=11)
    plt.ylabel("Query Tokens (Attending From)", fontsize=11)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

def plot_cls_attention(attentions, tokens, layer_idx=11):
    """Plots the attention distribution from [CLS] averaged across all heads in a layer."""
    # Shape: (12, seq_len, seq_len) -> take row 0 ([CLS] query), mean over heads
    cls_attn = attentions[layer_idx][0, :, 0, :].mean(dim=0).numpy()
    
    plt.figure(figsize=(10, 4))
    bars = plt.bar(range(len(tokens)), cls_attn, color="steelblue", edgecolor="black", alpha=0.85)
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right", fontsize=10)
    plt.title(f"Average [CLS] Attention Distribution across Heads (Layer {layer_idx + 1})", fontsize=13, pad=12)
    plt.xlabel("Tokens", fontsize=11)
    plt.ylabel("Attention Weight", fontsize=11)
    plt.ylim(0, max(cls_attn) * 1.25)
    
    # Label bars
    for bar in bars:
        yval = bar.get_height()
        if yval > 0.03:
            plt.text(bar.get_x() + bar.get_width() / 2.0, yval + 0.005, f"{yval:.2f}", ha='center', va='bottom', fontsize=9)
            
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

# 1. Plot full matrix for Layer 12, Head 1
plot_attention_matrix(cpu_attentions, tokens, layer_idx=11, head_idx=0)

# 2. Plot [CLS] attention in final layer (Layer 12)
plot_cls_attention(cpu_attentions, tokens, layer_idx=11)

## 6. Interactive BertViz Visualization (Optional)

BertViz provides interactive D3 visualization for head-level attention exploration.
We inject `require.js` first to ensure compatibility with modern JupyterLab and VS Code webviews.

In [ ]:
try:
    from bertviz import head_view
    head_view(cpu_attentions, tokens)
except Exception as e:
    print(f"BertViz interactive rendering skipped: {e}")
    print("You can rely on the static heatmaps above for analysis.")

## 7. Linguistic Interpretation & Scientific Caveats

- **Attention Sinks:** Tokens like `[SEP]` and `[CLS]` frequently receive high attention across middle layers. As shown by Clark et al. (*What Does BERT Look At?*), these act as default or "no-op" states when no strong lexical or syntactic dependency is triggered.
- **Discourse Connectives:** Coordinating conjunctions like `"but"` act as affective hinges, routing attention between opposing sentiment phrases (*"absolutely fantastic"* vs. *"acting was terrible"*).
- **Interpretability Caveat:** Attention weights reflect internal information routing through the transformer encoder; they do not constitute strict causal explanations of the model's decision, and raw softmax outputs are model probabilities rather than calibrated real-world confidence metrics.